# Fraud Detection
**OIBSIP Data Analytics — Level 2, Task 3**

Goal: classify transactions as legitimate (0) or potentially fraudulent (1). The included dataset is synthetic practice data.

## 1. Import libraries and load data

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
BASE = Path.cwd()
if not (BASE / "data/transactions.csv").exists() and (BASE / "Fraud_Detection/data/transactions.csv").exists(): BASE = BASE / "Fraud_Detection"
df = pd.read_csv(BASE / "data/transactions.csv")
df.head()


## 2. Inspect the dataset and check class balance

In [ ]:
print("Shape:", df.shape)
display(df.isna().sum().to_frame("Missing values"))
display(df["Is_Fraud"].value_counts().rename(index={0:"Legitimate",1:"Fraud"}).to_frame("Transactions"))
df.describe(include="all")


In [ ]:
df["Is_Fraud"].value_counts().sort_index().plot(kind="bar")
plt.xticks([0,1], ["Legitimate (0)", "Fraud (1)"], rotation=0)
plt.ylabel("Number of transactions"); plt.title("Class Distribution"); plt.show()


## 3. Prepare features and split data
We use a stratified split to keep the fraud proportion similar in train and test sets. Categorical fields are one-hot encoded.

In [ ]:
target = "Is_Fraud"
X, y = df.drop(columns=[target]), df[target]
numeric_features = ["Transaction_Amount_INR","Transaction_Hour","Days_Since_Last_Transaction","Account_Age_Months"]
categorical_features = ["Online_Transaction","International_Transaction","Merchant_Category"]
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))


## 4. Train a Random Forest classifier

In [ ]:
model = RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced", min_samples_leaf=2)
pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)


## 5. Evaluate results
Accuracy alone can be misleading when fraud is rare. Review precision, recall, F1-score, and the confusion matrix.

In [ ]:
metrics = {"Accuracy": accuracy_score(y_test, pred), "Precision_Fraud": precision_score(y_test, pred, zero_division=0), "Recall_Fraud": recall_score(y_test, pred, zero_division=0), "F1_Fraud": f1_score(y_test, pred, zero_division=0)}
pd.DataFrame([metrics])


In [ ]:
print(classification_report(y_test, pred, zero_division=0))
cm = confusion_matrix(y_test, pred, labels=[0,1])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate","Fraud"]).plot(values_format="d")
plt.title("Confusion Matrix"); plt.show()


## 6. Feature importance

In [ ]:
names = pipe.named_steps["preprocessor"].get_feature_names_out()
importances = pipe.named_steps["model"].feature_importances_
fi = pd.DataFrame({"Feature": names, "Importance": importances}).sort_values("Importance", ascending=False)
display(fi.head(12))
fi.head(12).sort_values("Importance").plot(kind="barh", x="Feature", y="Importance", legend=False)
plt.xlabel("Feature importance"); plt.title("Top Feature Importances"); plt.tight_layout(); plt.show()


## 7. Conclusion and limitations
- Fraud detection is an imbalanced classification problem.
- Precision tells us how many flagged transactions were actually fraud in the test set; recall tells us how many fraud cases were detected.
- A confusion matrix shows false positives and false negatives.
- Feature importance does not prove causation.

**Important:** This is synthetic practice data, not real banking data. Do not use this model for real fraud screening or financial decisions. Use an appropriately sourced public dataset if the evaluator requires real data.